# Tutorial 13 — Time-Dependent Flows and Neural Operators

**MM845 — Tópicos de Geometria III: AI for Geometry**
Paired with **Lecture 13: PINNs for Geometry II & Neural Operators**

---

Tutorial 12 solved *static* problems: one equation, one solution, no time. This one adds
the two ingredients the course has been building towards.

**Time.** A space-time PINN treats $t$ as just another input and learns the whole
trajectory at once (slide 3). That convenience hides a trap — nothing in the loss tells
the optimiser to learn the early dynamics *before* the late ones — and Lecture 13 spends
two slides on the cures.

**Geometric flows**, the prize targets of slide 4: curves and surfaces that move by their
own curvature. These are the equations this audience actually cares about, and they come
with exact laws we can check a network against without ever knowing the solution.

Then the change of perspective that closes the course: **neural operators** (slides 8–10)
learn a solution *map* between function spaces, reusable across a whole family of
problems, rather than one solution at a time.

| § | Question | Lecture 13 |
|---|---|---|
| 1 | How do we put time into a PINN, and what does a good ansatz buy? | slide 3 |
| 2 | Why does the long-horizon problem fail, and which cures work? | slides 6, 7 |
| 3 | Curve shortening flow, verified against exact geometric laws | slide 4 |
| 4 | Learning the solution operator instead of the solution | slides 8–10 |
| 5 | What to take away — and the course, assembled | slides 11, 12 |

You need the `aigeo` environment from [Tutorial 1](../tutorial_01/README.md).

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch

torch.set_default_dtype(torch.float64)
grad = torch.autograd.grad

SEED = 20261012
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

GEO_DARK, GEO_TEAL, GEO_RUST = "#103158", "#006c86", "#b2461e"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.titlesize": 10,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.prop_cycle": plt.cycler(color=[GEO_DARK, GEO_TEAL, GEO_RUST])})
print("torch", torch.__version__)

---
## 1. Time as one more input

The heat equation on the circle, $u_t = u_{xx}$ for $x \in [0, 2\pi)$, with

$$u_0(x) = \sin x + 0.6\sin 2x + 0.4\sin 3x + 0.3\sin 4x .$$

Each Fourier mode decays independently, $\hat u_k(t) = \hat u_k(0)e^{-k^2t}$, so we have an
exact solution to measure against.

A space-time PINN uses one network $u_\theta(x,t)$ trained on the residual
$u_t - u_{xx}$ at points scattered through the slab $[0,2\pi)\times[0,T]$. Time is an
input, not a loop, so a single trained network answers any query $(x,t)$ — and the initial
condition is simply the boundary $t=0$ of the space-time domain.

Tutorial 12's lesson was to **build constraints into the ansatz**. Two apply here:

- **The initial condition**, exactly: $u_\theta(x,t) = u_0(x) + t\,N_\theta(x,t)$ is equal
  to $u_0$ at $t = 0$ whatever the weights do.
- **Periodicity in $x$**, exactly: feed the network
  $(\cos kx, \sin kx)_{k=1}^{6}$ instead of $x$ itself. These are the eigenfunctions of
  $\Delta$ on the circle, so this is Lecture 13's Fourier-feature idea (slide 7) and
  Tutorial 7's spectral basis at the same time.

We compare that against the versions where each constraint is only a penalty.

In [ ]:
A_K = np.array([1.0, 0.6, 0.4, 0.3])
T_END = 0.4


def u0_torch(x):
    return sum(A_K[k - 1] * torch.sin(k * x) for k in range(1, len(A_K) + 1))


def u_exact(x, t):
    return sum(A_K[k - 1] * np.exp(-k**2 * t) * np.sin(k * x) for k in range(1, len(A_K) + 1))


def circle_features(x, K=6):
    '''Eigenfunctions of the Laplacian on the circle: periodicity for free.'''
    k = torch.arange(1, K + 1, dtype=x.dtype)
    return torch.cat([torch.cos(x * k), torch.sin(x * k)], dim=1)


def mlp(d_in, d_out=1, width=64, depth=3):
    layers, d = [], d_in
    for _ in range(depth):
        layers += [torch.nn.Linear(d, width), torch.nn.Tanh()]
        d = width
    return torch.nn.Sequential(*layers, torch.nn.Linear(d, d_out))


def train(loss_fn, params, adam=1000, lbfgs=200, lr=2e-3):
    '''Adam on freshly sampled points, then L-BFGS on one fixed set.'''
    opt = torch.optim.Adam(params, lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, adam)
    r = np.random.default_rng(0)
    t0 = time.time()
    for _ in range(adam):
        loss = loss_fn(r)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    lb = torch.optim.LBFGS(params, max_iter=lbfgs, line_search_fn="strong_wolfe")
    def closure():
        lb.zero_grad()
        loss = loss_fn(np.random.default_rng(7))      # the same points at every evaluation
        loss.backward()
        return loss
    lb.step(closure)
    return time.time() - t0


XG, TG = np.meshgrid(np.linspace(0, 2 * np.pi, 201), np.linspace(0, T_END, 101))
UG = u_exact(XG, TG)
XT, TT = torch.tensor(XG.ravel()[:, None]), torch.tensor(TG.ravel()[:, None])


def rel_l2(u_fn, ref=UG):
    with torch.no_grad():
        p = u_fn(XT, TT).numpy().ravel()
    return float(np.linalg.norm(p - ref.ravel()) / np.linalg.norm(ref.ravel()))


def sample(r, n, lo=0.0, hi=T_END, x_hi=2 * np.pi):
    x = torch.tensor(r.uniform(0, x_hi, (n, 1)), requires_grad=True)
    t = torch.tensor(r.uniform(lo, hi, (n, 1)), requires_grad=True)
    return x, t


def heat_residual(u_fn, x, t):
    u = u_fn(x, t)
    u_t = grad(u.sum(), t, create_graph=True)[0]
    u_x = grad(u.sum(), x, create_graph=True)[0]
    u_xx = grad(u_x.sum(), x, create_graph=True)[0]
    return u_t - u_xx

In [ ]:
IC_X = torch.tensor(np.linspace(0, 2 * np.pi, 200)[:, None], requires_grad=True)
IC_T = torch.zeros_like(IC_X, requires_grad=True)
BC_T = torch.tensor(np.linspace(0, T_END, 200)[:, None], requires_grad=True)
BC_L = torch.zeros_like(BC_T, requires_grad=True)
BC_R = torch.full_like(BC_T, 2 * np.pi).requires_grad_(True)

runs = {}
for label, feats, hard_ic in [("periodic features + hard IC", True, True),
                              ("periodic features + soft IC", True, False),
                              ("plain (x, t) + soft IC + soft BC", False, False)]:
    torch.manual_seed(0)
    net = mlp(13 if feats else 2)

    def u_fn(x, t, net=net, feats=feats, hard_ic=hard_ic):
        h = torch.cat([circle_features(x), t], 1) if feats else torch.cat([x, t], 1)
        return u0_torch(x) + t * net(h) if hard_ic else net(h)

    def loss_fn(r, u_fn=u_fn, feats=feats, hard_ic=hard_ic):
        L = (heat_residual(u_fn, *sample(r, 500))**2).mean()
        if not hard_ic:
            L = L + ((u_fn(IC_X, IC_T) - u0_torch(IC_X))**2).mean()
        if not feats:
            L = L + ((u_fn(BC_L, BC_T) - u_fn(BC_R, BC_T))**2).mean()
        return L

    dt = train(loss_fn, list(net.parameters()))
    runs[label] = {"u": u_fn, "t": dt, "err": rel_l2(u_fn)}
    print(f"{label:34s} {dt:5.1f}s   rel L2 = {runs[label]['err']:.2e}")

Both constraints pay, and the difference between the best and the worst version is more
than two orders of magnitude for the same architecture and the same budget.

Now two checks that do not need the exact solution — the kind you would run on a real
problem. The hard ansatz reproduces $u_0$ at $t=0$ to machine precision by construction.
And the heat equation dissipates energy: $\frac{d}{dt}\int u^2 = -2\int u_x^2 \le 0$, so
$E(t) = \int_0^{2\pi} u^2\,dx$ must decrease monotonically.

In [ ]:
best = runs["periodic features + hard IC"]["u"]
with torch.no_grad():
    P = best(XT, TT).numpy().reshape(UG.shape)
err_t = np.linalg.norm(P - UG, axis=1) / np.linalg.norm(UG, axis=1)
energy = (P**2).mean(1) * 2 * np.pi
energy_exact = (UG**2).mean(1) * 2 * np.pi
t_line = TG[:, 0]

print(f"error at t=0 (hard IC): {err_t[0]:.1e}")
print(f"energy {energy[0]:.4f} -> {energy[-1]:.4f}   (exact {energy_exact[0]:.4f} -> {energy_exact[-1]:.4f})")
print(f"energy monotonically decreasing: {bool(np.all(np.diff(energy) < 0))}")

fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.3))
im = axes[0].imshow(UG, origin="lower", aspect="auto", cmap="RdBu_r",
                    extent=[0, 2 * np.pi, 0, T_END], vmin=-2, vmax=2)
axes[0].set_title("exact $u(x,t)$"); axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$t$")
fig.colorbar(im, ax=axes[0], shrink=0.85)
im = axes[1].imshow(np.log10(np.abs(P - UG) + 1e-16), origin="lower", aspect="auto",
                    cmap="magma", extent=[0, 2 * np.pi, 0, T_END], vmin=-7)
axes[1].set_title(r"PINN error, $\log_{10}|u_\theta-u|$"); axes[1].set_xlabel("$x$")
fig.colorbar(im, ax=axes[1], shrink=0.85)
for ax in axes[:2]:
    ax.grid(False)
axes[2].plot(t_line, energy, color=GEO_TEAL, lw=2, label=r"PINN $E(t)$")
axes[2].plot(t_line, energy_exact, "--", color=GEO_DARK, lw=1.2, label="exact")
axes[2].set_xlabel("$t$"); axes[2].set_ylabel(r"$E(t)=\int u^2\,dx$")
axes[2].set_title("energy dissipation"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

> **Exercise 1 — the space-time slab.**
> (a) Replace the hard initial condition by a penalty $\lambda_0 L_0$ and sweep
> $\lambda_0 \in \{10^{-1},\dots,10^{3}\}$. Plot the error at $t = 0$ against the error at
> $t = T$. Which end of the slab does each weight buy?
>
> (b) **A discrete-time PINN** (slide 3's alternative). Fix a step $\Delta t$ and train one
> network per step to satisfy backward Euler,
> $u^{n+1} - \Delta t\,u^{n+1}_{xx} = u^{n}$, taking $u^n$ from the previous network.
> Compare accuracy and cost with the space-time version over the same horizon.
>
> (c) Drop the number of Fourier features from 6 to 2, so the basis can no longer represent
> $u_0$. Where does the error appear, and does it decay as the high modes die out?

---
## 2. Where it breaks: long horizons, and the cures

The heat equation was kind: it *forgets*. High-frequency detail decays, so the solution
at late times is smoother than at early times, and a network biased towards smooth
functions is biased in the right direction.

Transport is the opposite, and it is the standard laboratory for the failure Lecture 13
calls **propagation failure**. Take

$$u_t + c\,u_x = 0, \qquad c = 2, \qquad u(x,0) = u_0(x),$$

whose exact solution is $u(x,t) = u_0(x - ct)$: the initial profile rigidly translating,
never decaying. Longer horizon means more structure to represent — over a horizon $T$ the
network must hold $cT/2\pi$ full revolutions in its weights — and the residual alone gives
the optimiser no reason to get the early times right first.

Same architecture, same budget, three horizons.

In [ ]:
C_ADV = 2.0


def u0_adv(x):
    return torch.sin(x) + 0.5 * torch.sin(2 * x)


def u_exact_adv(x, t):
    y = x - C_ADV * t
    return np.sin(y) + 0.5 * np.sin(2 * y)


def adv_residual(u_fn, x, t):
    u = u_fn(x, t)
    u_t = grad(u.sum(), t, create_graph=True)[0]
    u_x = grad(u.sum(), x, create_graph=True)[0]
    return u_t + C_ADV * u_x


def adv_grids(T_end):
    xg, tg = np.meshgrid(np.linspace(0, 2 * np.pi, 257)[:-1], np.linspace(0, T_end, 101))
    return xg, tg, u_exact_adv(xg, tg)


def adv_err(u_fn, T_end):
    xg, tg, ref = adv_grids(T_end)
    with torch.no_grad():
        p = u_fn(torch.tensor(xg.ravel()[:, None]), torch.tensor(tg.ravel()[:, None])).numpy().ravel()
    return float(np.linalg.norm(p - ref.ravel()) / np.linalg.norm(ref.ravel())), p.reshape(ref.shape), ref


def vanilla_pinn(T_end, adam=800, seed=0):
    torch.manual_seed(seed)
    net = mlp(13)
    u_fn = lambda x, t: u0_adv(x) + t * net(torch.cat([circle_features(x), t], 1))
    dt = train(lambda r: (adv_residual(u_fn, *sample(r, 1000, 0.0, T_end))**2).mean(),
               list(net.parameters()), adam=adam)
    return u_fn, dt


horizons = {}
for T_end in (1.0, 2.0, 4.0):
    u_fn, dt = vanilla_pinn(T_end)
    e, P_adv, R_adv = adv_err(u_fn, T_end)
    horizons[T_end] = {"u": u_fn, "t": dt, "err": e, "P": P_adv, "ref": R_adv}
    print(f"horizon T = {T_end:.0f}  ({C_ADV * T_end / (2 * np.pi):.2f} revolutions)  "
          f"{dt:4.1f}s   rel L2 = {e:.2e}")

The same network that is accurate over one horizon is **useless** over four, and the
failure is not subtle: a relative error near $1$ means the prediction is no better than
zero. This is the pathology slide 3 warns about, and Lecture 13 offers three cures.

**Causal weighting** (slide 6). Split $[0,T]$ into ordered bins with mean squared
residuals $r_i$ and weight them by

$$w_i = \exp\Big(-\epsilon \sum_{j<i} r_j\Big),$$

so a bin only starts to matter once every earlier bin is already solved — the optimiser is
forced to work forwards in time. The weights are treated as constants (no gradient).

**Fourier features in time** (slide 7). Spectral bias is about *frequency*, and
transport puts the frequency in $t$ as much as in $x$. Giving the network
$(\cos m\pi t/T, \sin m\pi t/T)$ alongside the spatial features attacks it directly.

**Time-window marching** (slide 6). Cut $[0,T]$ into windows and train one small network
per window, each starting from the previous window's end state:
$u^{(w)}(x,t) = u^{(w-1)}(x, t_w) + (t - t_w)N_w(x, t - t_w)$. Every window is a
*short-horizon* problem, which is the regime that worked.

In [ ]:
T_HARD = 4.0
cures = {"vanilla": {"err": horizons[T_HARD]["err"], "t": horizons[T_HARD]["t"]}}

# ---- causal weighting
N_BINS, EPS_C = 20, 1.0
BIN_EDGES = torch.linspace(0, T_HARD, N_BINS + 1)
torch.manual_seed(0)
net_c = mlp(13)
u_causal = lambda x, t: u0_adv(x) + t * net_c(torch.cat([circle_features(x), t], 1))
weights_seen = {}


def loss_causal(r):
    x, t = sample(r, 1000, 0.0, T_HARD)
    r2 = (adv_residual(u_causal, x, t)**2).squeeze(1)
    idx = torch.bucketize(t.squeeze(1).detach(), BIN_EDGES[1:-1])
    total = torch.zeros(N_BINS).index_add_(0, idx, r2)
    count = torch.zeros(N_BINS).index_add_(0, idx, torch.ones_like(r2)).clamp(min=1)
    per_bin = total / count
    with torch.no_grad():                                   # weights carry no gradient
        w = torch.exp(-EPS_C * torch.cat([torch.zeros(1), torch.cumsum(per_bin, 0)[:-1]]))
        weights_seen["w"] = w.numpy()
    return (w * per_bin).mean()


cures["causal weighting"] = {"t": train(loss_causal, list(net_c.parameters()), adam=800)}
cures["causal weighting"]["err"] = adv_err(u_causal, T_HARD)[0]

# ---- Fourier features in time as well as space
def space_time_features(x, t, K=6, M=6):
    kx = torch.arange(1, K + 1, dtype=x.dtype)
    kt = torch.arange(1, M + 1, dtype=x.dtype) * (np.pi / T_HARD)
    return torch.cat([torch.cos(x * kx), torch.sin(x * kx),
                      torch.cos(t * kt), torch.sin(t * kt), t], dim=1)


torch.manual_seed(0)
net_f = mlp(25)
u_feat = lambda x, t: u0_adv(x) + t * net_f(space_time_features(x, t))
cures["time features"] = {"t": train(lambda r: (adv_residual(u_feat, *sample(r, 1000, 0.0, T_HARD))**2).mean(),
                                     list(net_f.parameters()), adam=800)}
cures["time features"]["err"] = adv_err(u_feat, T_HARD)[0]

# ---- time-window marching
N_WIN = 4
W_EDGES = np.linspace(0, T_HARD, N_WIN + 1)
win_nets = []
for w in range(N_WIN):
    torch.manual_seed(w)
    win_nets.append(mlp(13))


def u_window(w, x, t):
    '''Window w starts from window w-1 evaluated at the interface time.'''
    lo = W_EDGES[w]
    base = u0_adv(x) if w == 0 else u_window(w - 1, x, torch.full_like(x, lo))
    d = t - lo
    return base + d * win_nets[w](torch.cat([circle_features(x), d], 1))


t0 = time.time()
for w in range(N_WIN):
    train(lambda r, w=w: (adv_residual(lambda x, t: u_window(w, x, t),
                                       *sample(r, 800, W_EDGES[w], W_EDGES[w + 1]))**2).mean(),
          list(win_nets[w].parameters()), adam=400, lbfgs=150)
t_march = time.time() - t0


def u_marched(x, t):
    tn = t.detach().numpy().ravel()
    idx = np.clip(np.searchsorted(W_EDGES, tn, side="right") - 1, 0, N_WIN - 1)
    out = torch.zeros(len(tn), 1)
    for w in range(N_WIN):
        m = np.flatnonzero(idx == w)
        if len(m):
            mm = torch.tensor(m)
            out[mm] = u_window(w, x[mm], t[mm])
    return out


cures["window marching"] = {"t": t_march, "err": adv_err(u_marched, T_HARD)[0]}

print(f"horizon T = {T_HARD:.0f}\n{'':20s} {'time':>7s} {'rel L2':>10s}")
for k, v in cures.items():
    print(f"{k:20s} {v['t']:6.1f}s {v['err']:10.2e}")
print(f"\ncausal weights at the end of training: min {weights_seen['w'].min():.2f}, "
      f"max {weights_seen['w'].max():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.3))

ax = axes[0]
Ts = sorted(horizons)
ax.semilogy(Ts, [horizons[T]["err"] for T in Ts], "o-", color=GEO_DARK)
ax.set_xlabel("horizon $T$"); ax.set_ylabel("relative $L^2$ error")
ax.set_title("one network, growing horizon"); ax.set_xticks(Ts)

ax = axes[1]
P4, R4 = horizons[T_HARD]["P"], horizons[T_HARD]["ref"]
tl = np.linspace(0, T_HARD, P4.shape[0])
e_van = np.linalg.norm(P4 - R4, axis=1) / np.linalg.norm(R4, axis=1)
ax.plot(tl[1:], e_van[1:], color=GEO_RUST, lw=1.6, label="vanilla")
with torch.no_grad():
    xg, tg, ref = adv_grids(T_HARD)
    pm = u_marched(torch.tensor(xg.ravel()[:, None]), torch.tensor(tg.ravel()[:, None])).numpy().reshape(ref.shape)
e_mar = np.linalg.norm(pm - ref, axis=1) / np.linalg.norm(ref, axis=1)
ax.plot(tl[1:], e_mar[1:], color=GEO_TEAL, lw=1.6, label="window marching")
ax.set_yscale("log"); ax.set_xlabel("$t$"); ax.set_ylabel("relative error at time $t$")
ax.set_title("error grows into the future"); ax.legend(fontsize=8)
print("relative error at t = 1, 2, 3, 4:")
for nm, e in [("vanilla", e_van), ("marching", e_mar)]:
    print(f"   {nm:9s} " + ", ".join(f"{e[int(q * 25)]:.2e}" for q in (1, 2, 3, 4)))

ax = axes[2]
names = list(cures)
ax.barh(names, [cures[k]["err"] for k in names],
        color=[GEO_RUST, GEO_DARK, GEO_TEAL, GEO_TEAL])
ax.set_xscale("log"); ax.set_xlabel(f"relative $L^2$ error at $T={T_HARD:.0f}$")
ax.set_xlim(0.05, 1.5); ax.set_xticks([0.1, 0.3, 1.0])
ax.set_xticklabels(["0.1", "0.3", "1.0"]); ax.minorticks_off()
ax.set_title("the cures"); ax.invert_yaxis(); ax.grid(axis="y")
plt.tight_layout(); plt.show()

Both structural cures work, and they land within a factor of two of each other; the plain
reweighting is much the weakest of the three here.

The middle panel shows the mechanism. The vanilla network's error *grows into the future* —
already 65% at $t=1$ and near 100% by $t=4$ — which is precisely the propagation failure of
slide 3: nothing made it solve the early times first. Marching grows too, since each window
inherits the previous window's error, but it starts an order of magnitude lower and stays
five to eight times below the vanilla run at every time, because each window only ever faces
a *short-horizon* problem.

Read the causal weights before drawing a general conclusion. If they stay close to $1$,
the gate never really closed and causal training reduced to the vanilla loss with a
slightly noisier objective; the parameter $\epsilon$ has to be tuned per problem, and the
cost of setting it too high is that late times receive no gradient at all and drift
freely. Time marching has no such knob: it changes the *problem* rather than the weights.

> **Exercise 2 — tuning the cures.**
> (a) Sweep $\epsilon \in \{10^{-2},1,10^2\}$ and print the final weights $w_i$ alongside
> the error. Find a value that genuinely gates, and check whether it beats marching.
>
> (b) **Gradient balancing** (slide 6). For a soft-constraint version, record
> $\lVert\nabla_\theta L_{\text{res}}\rVert$ and $\lVert\nabla_\theta L_{0}\rVert$ during
> training and set $\lambda_0$ to equalise them. Compare with your best fixed weight.
>
> (c) Push to $T = 8$ with 4, 8 and 16 windows. Plot error against window count and
> against total training time; the pattern tells you what the window length has to match.
>
> (d) **FBPINN** (slide 7). Replace the disjoint windows by overlapping ones with a
> partition of unity $\sum_i w_i = 1$, blending the local networks. Does the interface
> behaviour change?

---
## 3. Curve shortening flow

Now the geometry. A closed curve $X(p,t)\in\mathbb R^2$ evolves by **curve shortening
flow** when every point moves with velocity equal to the curvature vector:

$$\partial_t X \;=\; \partial_s^2 X \;=\; \kappa\,\nu ,$$

with $s$ arclength — the one-dimensional case of slide 4's mean curvature flow
$\partial_t X = \Delta_{g(t)}X$. The geometry sits inside the operator: $\partial_s$ is
defined through the metric the curve itself induces, $|X_p|$, which changes as the curve
moves. In a PINN that costs nothing extra, because autodiff differentiates through it:

$$\partial_s^2 X = \frac{1}{|X_p|}\,\partial_p\!\left(\frac{X_p}{|X_p|}\right).$$

The ansatz $X_\theta(p,t) = X_0(p) + t\,N_\theta(p,t)$ with periodic features in $p$ starts
from the given curve exactly, as in §1.

**Why this flow is worth the trouble to check.** Two exact laws hold for *any* embedded
closed curve, and neither needs the solution:

- A circle of radius $R_0$ stays a circle, with $R(t) = \sqrt{R_0^2 - 2t}$ — so it vanishes
  at $t = R_0^2/2$.
- The enclosed area obeys $\dfrac{dA}{dt} = -\displaystyle\oint \kappa\,ds = -2\pi$ by the
  turning-number theorem. **Area decreases at a constant rate, whatever the shape.**

We check the circle against the first law and an ellipse against the second.

In [ ]:
def curve_features(p, K=8):
    k = torch.arange(1, K + 1, dtype=p.dtype)
    return torch.cat([torch.cos(p * k), torch.sin(p * k)], dim=1)


def flow_residual(X_fn, p, t):
    '''d_t X - d_ss X for a plane curve, entirely by autodiff.'''
    X = X_fn(p, t)
    X_t = torch.stack([grad(X[:, i].sum(), t, create_graph=True)[0][:, 0] for i in range(2)], 1)
    X_p = torch.stack([grad(X[:, i].sum(), p, create_graph=True)[0][:, 0] for i in range(2)], 1)
    speed = X_p.norm(dim=1, keepdim=True)
    tangent = X_p / speed
    T_p = torch.stack([grad(tangent[:, i].sum(), p, create_graph=True)[0][:, 0] for i in range(2)], 1)
    return X_t - T_p / speed


def sample_curve(r, n, lo, hi):
    p = torch.tensor(r.uniform(0, 2 * np.pi, (n, 1)), requires_grad=True)
    t = torch.tensor(r.uniform(lo, hi, (n, 1)), requires_grad=True)
    return p, t


def train_flow(X0_fn, T_end, adam=1200, lbfgs=200, n=500, seed=0):
    torch.manual_seed(seed)
    net = mlp(17, d_out=2)
    X_fn = lambda p, t: X0_fn(p) + t * net(torch.cat([curve_features(p), t], 1))
    dt = train(lambda r: (flow_residual(X_fn, *sample_curve(r, n, 0.0, T_end))**2).mean(),
               list(net.parameters()), adam=adam, lbfgs=lbfgs)
    return X_fn, dt


P_QUAD = torch.tensor(np.linspace(0, 2 * np.pi, 513)[:-1, None], requires_grad=True)


def curve_geometry(X_fn, t_val):
    '''Length, enclosed area and isoperimetric ratio, by spectral quadrature in p.'''
    t = torch.full_like(P_QUAD, t_val, requires_grad=True)
    X = X_fn(P_QUAD, t)
    X_p = torch.stack([grad(X[:, i].sum(), P_QUAD, create_graph=True)[0][:, 0] for i in range(2)], 1)
    dp = 2 * np.pi / len(P_QUAD)
    L = float((X_p.norm(dim=1).sum() * dp).detach())
    A = float((0.5 * (X[:, 0] * X_p[:, 1] - X[:, 1] * X_p[:, 0]).sum() * dp).detach())
    return L, A, L**2 / (4 * np.pi * A)


circle0 = lambda p: torch.cat([torch.cos(p), torch.sin(p)], 1)
circ = {}
for T_end in (0.3, 0.45):                    # the circle vanishes at t = 0.5
    X_fn, dt = train_flow(circle0, T_end)
    ts = np.linspace(0, T_end, 10)
    R = [float(X_fn(P_QUAD, torch.full_like(P_QUAD, tv)).norm(dim=1).mean().detach()) for tv in ts]
    R_exact = np.sqrt(1 - 2 * ts)
    circ[T_end] = {"X": X_fn, "ts": ts, "R": np.array(R), "Rx": R_exact, "t": dt}
    worst = np.max(np.abs(np.array(R) - R_exact) / R_exact)
    print(f"circle to T={T_end} ({T_end / 0.5:.0%} of the way to extinction)  {dt:4.1f}s   "
          f"max relative radius error {worst:.2e}   R(T) = {R[-1]:.4f} vs {R_exact[-1]:.4f}")

The same network and the same budget, two horizons: under 1% error while the curve is
comfortably alive, and around 50% once we ask it to run to 90% of the extinction time. A
factor of fifty, bought by nothing but moving the finish line. That is slide 4's warning
made quantitative — **the singularity is the hard part**. The curvature blows up like $1/R$, the solution
develops the sharpest features exactly where the network has the least data, and nothing
in the loss knows the curve is about to disappear. The lecture's remedies (rescaling near
the singularity, adaptive sampling, weak formulations) all exist for this.

For a curve with no closed-form solution we use the area law instead, and §2's cure: three
time windows, each starting from the previous one's end state. The classical baseline is a
polygon that moves by its discrete curvature, with points redistributed by arclength each
step — the same comparison Tutorial 12 insisted on.

In [ ]:
A_ELL, B_ELL, T_ELL = 1.5, 0.7, 0.3
ellipse0 = lambda p: torch.cat([A_ELL * torch.cos(p), B_ELL * torch.sin(p)], 1)
AREA0 = np.pi * A_ELL * B_ELL
print(f"ellipse: A0 = {AREA0:.4f}, so the area law predicts extinction at "
      f"t = A0/(2 pi) = {AREA0 / (2 * np.pi):.4f}")


def polygon_flow(T_end, n_steps=20000, n_pts=256):
    '''Classical baseline: explicit curvature flow of a polygon, reparametrised each step.'''
    th = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
    P = np.stack([A_ELL * np.cos(th), B_ELL * np.sin(th)], 1)
    dt, traj, ts = T_end / n_steps, [P.copy()], [0.0]
    for n in range(1, n_steps + 1):
        Pm, Pp = np.roll(P, 1, 0), np.roll(P, -1, 0)
        dm = np.linalg.norm(P - Pm, axis=1, keepdims=True)
        dp = np.linalg.norm(Pp - P, axis=1, keepdims=True)
        P = P + dt * 2 * ((Pp - P) / dp - (P - Pm) / dm) / (dm + dp)
        d = np.linalg.norm(np.roll(P, -1, 0) - P, axis=1)
        s = np.concatenate([[0], np.cumsum(d)])
        Pc = np.vstack([P, P[:1]])
        tgt = np.linspace(0, s[-1], n_pts, endpoint=False)
        P = np.stack([np.interp(tgt, s, Pc[:, 0]), np.interp(tgt, s, Pc[:, 1])], 1)
        if n % (n_steps // 6) == 0:
            traj.append(P.copy()); ts.append(n * dt)
    return np.array(ts), traj


t0 = time.time()
TS_POLY, TRAJ_POLY = polygon_flow(T_ELL)
T_POLY = time.time() - t0


def poly_geometry(P):
    Pp = np.roll(P, -1, 0)
    L = np.linalg.norm(Pp - P, axis=1).sum()
    A = 0.5 * abs(np.sum(P[:, 0] * Pp[:, 1] - Pp[:, 0] * P[:, 1]))
    return L, A, L**2 / (4 * np.pi * A)


# ---- windowed PINN for the ellipse
N_WE = 3
E_EDGES = np.linspace(0, T_ELL, N_WE + 1)
ell_nets = []
for w in range(N_WE):
    torch.manual_seed(w)
    ell_nets.append(mlp(17, d_out=2))


def X_window(w, p, t):
    lo = E_EDGES[w]
    base = ellipse0(p) if w == 0 else X_window(w - 1, p, torch.full_like(p, lo))
    d = t - lo
    return base + d * ell_nets[w](torch.cat([curve_features(p), d], 1))


t0 = time.time()
for w in range(N_WE):
    train(lambda r, w=w: (flow_residual(lambda p, t: X_window(w, p, t),
                                        *sample_curve(r, 400, E_EDGES[w], E_EDGES[w + 1]))**2).mean(),
          list(ell_nets[w].parameters()), adam=400, lbfgs=150)
T_PINN_ELL = time.time() - t0


def X_ellipse(p, t):
    tv = float(t.detach().reshape(-1)[0])
    w = int(np.clip(np.searchsorted(E_EDGES, tv, side="right") - 1, 0, N_WE - 1))
    return X_window(w, p, t)


ts_e = np.linspace(0, T_ELL, 7)
geo_pinn = np.array([curve_geometry(X_ellipse, tv) for tv in ts_e])
A_LAW = AREA0 - 2 * np.pi * ts_e
geo_poly = np.array([poly_geometry(P) for P in TRAJ_POLY])

print(f"\nPINN (3 windows) {T_PINN_ELL:.1f}s    polygon baseline {T_POLY:.2f}s\n")
print(f"{'t':>6s} {'A (PINN)':>10s} {'A (polygon)':>12s} {'A0 - 2 pi t':>12s} {'iso (PINN)':>11s}")
for i, tv in enumerate(ts_e):
    print(f"{tv:6.2f} {geo_pinn[i, 1]:10.4f} {geo_poly[i, 1]:12.4f} {A_LAW[i]:12.4f} {geo_pinn[i, 2]:11.4f}")
print(f"\nfitted dA/dt: PINN {np.polyfit(ts_e, geo_pinn[:, 1], 1)[0]:.4f}, "
      f"polygon {np.polyfit(TS_POLY, geo_poly[:, 1], 1)[0]:.4f}, exact {-2 * np.pi:.4f}")
print(f"max |A - (A0 - 2 pi t)|: PINN {np.abs(geo_pinn[:, 1] - A_LAW).max():.2e}, "
      f"polygon {np.abs(geo_poly[:, 1] - (AREA0 - 2 * np.pi * TS_POLY)).max():.2e}")
print(f"isoperimetric ratio {geo_pinn[0, 2]:.4f} -> {geo_pinn[-1, 2]:.4f}  (a circle has 1)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.5))

ax = axes[0]
for tv, col in zip(np.linspace(0, T_ELL, 4), [GEO_DARK, GEO_TEAL, GEO_RUST, "black"]):
    with torch.no_grad():
        Xc = X_ellipse(P_QUAD, torch.full_like(P_QUAD, tv)).numpy()
    ax.plot(*np.vstack([Xc, Xc[:1]]).T, color=col, lw=1.8, label=f"$t={tv:.1f}$")
    j = int(round(tv / T_ELL * (len(TRAJ_POLY) - 1)))
    Pp = TRAJ_POLY[min(j, len(TRAJ_POLY) - 1)]
    ax.plot(*np.vstack([Pp, Pp[:1]]).T, "--", color=col, lw=1.0)
ax.set_aspect("equal"); ax.set_title("ellipse under curve shortening\n(dashed: polygon baseline)")
ax.legend(fontsize=7, loc="upper right")

ax = axes[1]
ax.plot(ts_e, geo_pinn[:, 1], "o-", color=GEO_TEAL, label="PINN")
ax.plot(TS_POLY, geo_poly[:, 1], "s--", color=GEO_DARK, ms=4, label="polygon")
ax.plot(ts_e, A_LAW, ":", color=GEO_RUST, lw=2, label=r"exact law $A_0-2\pi t$")
ax.set_xlabel("$t$"); ax.set_ylabel("enclosed area"); ax.set_title(r"$dA/dt=-2\pi$, whatever the shape")
ax.legend(fontsize=8)

ax = axes[2]
t_fine = np.linspace(0, 0.45, 200)
ax.plot(t_fine, np.sqrt(1 - 2 * t_fine), "--", color="black", lw=1.2, label="exact")
for T_end, col in [(0.3, GEO_TEAL), (0.45, GEO_RUST)]:
    d = circ[T_end]
    ax.plot(d["ts"], d["R"], "o-", color=col, ms=4, label=f"PINN trained to $T={T_end}$")
ax.set_xlabel("$t$"); ax.set_ylabel("radius")
ax.set_title(r"circle: $R(t)=\sqrt{1-2t}$ (dashed)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

The PINN tracks the area law and stays close to the classical polygon, and the
isoperimetric ratio falls towards $1$: the ellipse is becoming round, which is the
Gage–Hamilton–Grayson theorem showing up in a numerical experiment. The classical solver
is again far cheaper — the honest verdict of Tutorial 12 survives into time-dependent
problems.

What the PINN gives instead is a *continuous* object. $X_\theta$ is defined at every
$(p,t)$, differentiable in both, with no time step and no mesh; the same code with
$\Delta_{g(t)}X$ in three dimensions is mean curvature flow of a surface, where a
classical implementation is a much larger undertaking.

One detail from slide 4 worth noticing: we solved $\partial_t X = \partial_s^2X$, whose
tangential part reparametrises the curve while the normal part moves it. Only the normal
motion is geometric, but the tangential part is what keeps the parametrisation healthy.

> **Exercise 3 — flows and their traps.**
> (a) Check how equally the flow spreads the parametrisation: plot $|X_p|(p)$ at $t=0$ and
> at $t=T$ for the ellipse. Then solve the purely normal flow
> $\partial_t X = (\partial_s^2X\cdot\nu)\,\nu$ instead and compare.
>
> (b) **Non-convex initial data.** Start from $r(\theta) = 1 + 0.35\cos 3\theta$ and watch
> the flow become convex, then round. Verify that $dA/dt = -2\pi$ still holds, and find the
> time at which the curvature first becomes positive everywhere.
>
> (c) **The level-set formulation** (slide 4). Represent the curve as $\{\phi_\theta = 0\}$
> and fit $\partial_t\phi = |\nabla\phi|\operatorname{div}(\nabla\phi/|\nabla\phi|)$.
> Start from two nearby circles and check whether the merge happens. What does the
> parametric formulation do with the same initial data?
>
> (d) **Rescale the singularity.** For the circle, train in the variable
> $\tau = -\log(R_0^2 - 2t)$ so that extinction is pushed to $\tau=\infty$. How far towards
> extinction can you get?

---
## 4. Learning the operator, not the solution

Every network so far solved **one** problem. Change the initial condition and all the work
is gone. A **neural operator** (slide 8) learns the solution map itself,

$$G: u_0 \longmapsto u(\cdot,T),$$

from many input/output pairs, so that a new instance costs a forward pass instead of a
training run.

We take the heat semigroup from §1 at $T = 0.05$. The training data can be generated
*exactly*, since each Fourier mode decays by $e^{-k^2T}$, with no solver involved. Inputs
are random trigonometric polynomials with modes $k \le 8$ and coefficients of size $1/k$,
sampled on a grid of 64 points.

Two architectures from the lecture:

- **DeepONet** (slide 8): a *branch* net reads the input function at the 64 grid points and
  produces coefficients; a *trunk* net reads the query point $y$ and produces a basis;
  the prediction is $\sum_k b_k(u_0)\,\tau_k(y)$.
- **A spectral layer**, the heart of the **FNO** (slide 9): transform to Fourier
  coefficients, multiply the retained low modes by *learned* weights $R_k$, transform back.

On the circle, the Laplacian's eigenfunctions **are** the Fourier modes, so this spectral
layer is exactly slide 10's geometric operator learning in the simplest possible
geometry — and the true operator is a Fourier multiplier, so we can check the learned
weights against $e^{-k^2T}$ one mode at a time.

In [ ]:
K_DATA, T_OP, N_GRID = 8, 0.05, 64


def random_coeffs(n, r, kmax=K_DATA):
    k = np.arange(1, kmax + 1)
    return r.normal(size=(n, kmax)) / k, r.normal(size=(n, kmax)) / k


def evaluate(a, b, x, t):
    '''Exact heat evolution of a trigonometric polynomial.'''
    k = np.arange(1, a.shape[1] + 1)
    decay = np.exp(-k**2 * t)
    return (a * decay) @ np.sin(np.outer(k, x)) + (b * decay) @ np.cos(np.outer(k, x))


X_OP = np.linspace(0, 2 * np.pi, N_GRID, endpoint=False)
a_tr, b_tr = random_coeffs(1000, np.random.default_rng(0))
a_te, b_te = random_coeffs(200, np.random.default_rng(1))
U0_TR, UT_TR = evaluate(a_tr, b_tr, X_OP, 0.0), evaluate(a_tr, b_tr, X_OP, T_OP)
U0_TE, UT_TE = evaluate(a_te, b_te, X_OP, 0.0), evaluate(a_te, b_te, X_OP, T_OP)
U0_t, UT_t = torch.tensor(U0_TR), torch.tensor(UT_TR)
rel = lambda p, r: float(np.linalg.norm(p - r) / np.linalg.norm(r))
print(f"{U0_TR.shape[0]} training pairs, {U0_TE.shape[0]} test pairs, generated exactly")


def fit(params, step_fn, steps, lr):
    opt = torch.optim.Adam(params, lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    r = np.random.default_rng(0)
    t0 = time.time()
    for _ in range(steps):
        idx = torch.tensor(r.integers(0, len(U0_t), 64))
        loss = step_fn(idx)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return time.time() - t0


# ---- DeepONet
P_BASIS = 32
torch.manual_seed(0)
branch, trunk = mlp(N_GRID, P_BASIS), mlp(12, P_BASIS)
FY = circle_features(torch.tensor(X_OP[:, None]))
deeponet = lambda u0: branch(u0) @ trunk(FY).T
t_don = fit(list(branch.parameters()) + list(trunk.parameters()),
            lambda idx: ((deeponet(U0_t[idx]) - UT_t[idx])**2).mean(), 2000, 2e-3)
with torch.no_grad():
    pred_don = deeponet(torch.tensor(U0_TE)).numpy()
n_don = sum(p.numel() for p in list(branch.parameters()) + list(trunk.parameters()))
print(f"DeepONet       {n_don:6d} params  {t_don:4.1f}s   test rel L2 = {rel(pred_don, UT_TE):.2e}")


# ---- one spectral (FNO) layer
class SpectralLayer(torch.nn.Module):
    '''Keep the lowest modes and learn one complex multiplier for each.'''
    def __init__(self, modes=K_DATA + 2):
        super().__init__()
        self.modes = modes
        self.re = torch.nn.Parameter(torch.ones(modes))
        self.im = torch.nn.Parameter(torch.zeros(modes))

    def forward(self, u):
        u_hat = torch.fft.rfft(u, dim=-1)
        out = torch.zeros_like(u_hat)
        out[..., :self.modes] = u_hat[..., :self.modes] * torch.complex(self.re, self.im)
        return torch.fft.irfft(out, n=u.shape[-1], dim=-1)


torch.manual_seed(0)
spectral = SpectralLayer()
t_sp = fit(list(spectral.parameters()),
           lambda idx: ((spectral(U0_t[idx]) - UT_t[idx])**2).mean(), 600, 5e-2)
with torch.no_grad():
    pred_sp = spectral(torch.tensor(U0_TE)).numpy()
    learned = torch.complex(spectral.re, spectral.im).numpy()
n_sp = sum(p.numel() for p in spectral.parameters())
print(f"spectral layer {n_sp:6d} params  {t_sp:4.1f}s   test rel L2 = {rel(pred_sp, UT_TE):.2e}")

print("\nlearned multiplier against the exact heat kernel:")
print(f"{'k':>3s} {'|R_k| learned':>15s} {'exp(-k^2 T)':>13s} {'in training data':>18s}")
for k in (1, 2, 4, 8, 9):
    print(f"{k:3d} {abs(learned[k]):15.6f} {np.exp(-k**2 * T_OP):13.6f} {str(k <= K_DATA):>18s}")

In [ ]:
# discretisation invariance: the same weights on a grid never seen in training
print("same weights, finer grids (trained on 64 points):")
for N2 in (128, 256):
    x2 = np.linspace(0, 2 * np.pi, N2, endpoint=False)
    u0_2, uT_2 = evaluate(a_te, b_te, x2, 0.0), evaluate(a_te, b_te, x2, T_OP)
    with torch.no_grad():
        p2 = spectral(torch.tensor(u0_2)).numpy()
    print(f"   {N2:3d} points: rel L2 = {rel(p2, uT_2):.2e}")

# out of distribution: modes the training set never contained
a_od, b_od = random_coeffs(200, np.random.default_rng(2), kmax=14)
U0_OD, UT_OD = evaluate(a_od, b_od, X_OP, 0.0), evaluate(a_od, b_od, X_OP, T_OP)
with torch.no_grad():
    od_sp = spectral(torch.tensor(U0_OD)).numpy()
    od_don = deeponet(torch.tensor(U0_OD)).numpy()
print(f"\ninputs with modes up to 14 (trained on modes up to {K_DATA}):")
print(f"   spectral layer rel L2 = {rel(od_sp, UT_OD):.2e}   DeepONet rel L2 = {rel(od_don, UT_OD):.2e}")

fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.3))
ax = axes[0]
ks = np.arange(len(learned))
ax.semilogy(ks, np.abs(learned), "o", color=GEO_TEAL, ms=7, label="learned $|R_k|$")
ax.semilogy(ks, np.exp(-ks**2 * T_OP), "x", color=GEO_DARK, ms=8, label=r"exact $e^{-k^2T}$")
ax.axvspan(K_DATA + 0.5, ks[-1] + 0.5, color=GEO_RUST, alpha=0.12)
ax.text(K_DATA + 0.7, 3e-1, "never excited\nby the data", fontsize=7, color=GEO_RUST)
ax.set_xlabel("mode $k$"); ax.set_ylabel("multiplier"); ax.legend(fontsize=8)
ax.set_title("the operator, one mode at a time")

ax = axes[1]
j = 0
xf = np.linspace(0, 2 * np.pi, 400)
ax.plot(xf, evaluate(a_te[j:j + 1], b_te[j:j + 1], xf, 0.0)[0], color=GEO_DARK, lw=1.4, label="$u_0$")
ax.plot(xf, evaluate(a_te[j:j + 1], b_te[j:j + 1], xf, T_OP)[0], color=GEO_RUST, lw=2, label="$u(\\cdot,T)$ exact")
ax.plot(X_OP, pred_sp[j], "o", color=GEO_TEAL, ms=4, label="spectral layer")
ax.plot(X_OP, pred_don[j], "s", color="black", ms=3, alpha=0.6, label="DeepONet")
ax.set_xlabel("$x$"); ax.set_title("one held-out input function"); ax.legend(fontsize=7)

ax = axes[2]
labels = ["DeepONet", "spectral"]
inside = [rel(pred_don, UT_TE), rel(pred_sp, UT_TE)]
outside = [rel(od_don, UT_OD), rel(od_sp, UT_OD)]
y = np.arange(2)
ax.barh(y - 0.2, inside, 0.4, color=GEO_TEAL, label="modes $\\leq 8$ (trained)")
ax.barh(y + 0.2, outside, 0.4, color=GEO_RUST, label="modes $\\leq 14$ (unseen)")
ax.set_yticks(y); ax.set_yticklabels(labels); ax.set_xscale("log")
ax.set_xlabel("relative $L^2$ error"); ax.set_title("inside and outside the training family")
ax.legend(fontsize=7); ax.grid(axis="y")
plt.tight_layout(); plt.show()

Three things to take from this.

**A matched architecture is worth thousands of parameters.** The spectral layer learns the
heat semigroup essentially exactly, with twenty numbers, because the true operator *is* a
Fourier multiplier and the layer's hypothesis space contains it. DeepONet, which assumes
nothing about the geometry, needs three orders of magnitude more parameters and still does
far worse. That is not a fair fight, and it is the point: this is Lecture 10's argument
about symmetry, now about the operator's structure.

**Discretisation invariance is real** (slide 9). The learned multipliers are attached to
Fourier modes, not to grid points, so the same weights run on grids the training never
saw, at the same accuracy.

**But you only learn what the data excites.** The multiplier for mode $9$ never received a
gradient and sits at its initial value, so inputs containing that mode are evolved with a
wrong operator. A finer grid does not repair it. That is slide 9's caveat and slide 11's
verification discipline: an operator is accurate on the family it was trained on, and
*nothing* guarantees it beyond.

Against a PINN the trade is the one slide 9 describes as **amortisation**: §1 spent seconds
training one network for one initial condition, while this operator answers any new input
instantly — but it needed a thousand solved examples first, and each one came from a
classical solver in a real problem.

> **Exercise 4 — operators.**
> (a) Retrain the spectral layer on data containing modes up to 14 and confirm the
> previously untouched multipliers converge to $e^{-k^2T}$. How many training functions do
> you need before every mode is pinned down?
>
> (b) **Nonlinear operators.** Learn $u_0 \mapsto u(\cdot,T)$ for $u_t = u_{xx} + u^2$,
> generating data with the spectral solver of your choice. A single multiplier layer can no
> longer be exact — add the pointwise $Wv$ term and a nonlinearity, and see how much of the
> gap closes.
>
> (c) **Physics-informed operator** (slide 9). Train the spectral layer with *no* data, by
> penalising the PDE residual of $t \mapsto G_\theta^{(t)}u_0$ instead. How close does it get?
>
> (d) **A parametric family** (slide 10). Learn $\alpha \mapsto u_\alpha$ for
> $-\Delta u = f_\alpha$ on Tutorial 12's disk, with $\alpha$ a few geometric parameters,
> and compare a sweep of 100 values against 100 separate PINN solves.

---
## 5. What to take away — and the course, assembled

From this tutorial:

- **Time is an input, but the optimiser does not know time flows.** A space-time residual
  treats every instant alike, so nothing makes the network learn the early dynamics first.
- **Build the initial condition into the ansatz**, exactly as you build boundary conditions
  into a static PINN; on a periodic domain use the eigenfunctions of the Laplacian.
- **The horizon is the difficulty.** The same network over four times the horizon was about
  fifty times worse. Cut the horizon into windows and march, and each piece is easy again.
- **Geometric flows are learnable, and checkable.** Curve shortening comes with exact laws —
  $R(t)=\sqrt{R_0^2-2t}$ and $dA/dt = -2\pi$ — that verify a network with no reference
  solution. Use them; every geometric problem has invariants of this kind.
- **Singularities are the honest limit.** Run to 90% of the extinction time instead of 60%
  and the same network, at the same cost, was fifty times less accurate.
- **Operators amortise.** Learning $u_0 \mapsto u(\cdot,T)$ pays for itself over a family,
  transfers across discretisations — and holds only over the family it was trained on.

### The toolkit, assembled

Thirteen lectures, one framework. Facing a geometric problem, ask:

| Your task | Methods | Tutorials |
|---|---|---|
| **Predict** a quantity from examples | linear models, MLPs, CNNs, transformers, graph networks | 3, 4, 5, 6, 10 |
| **Discover** structure in examples | clustering, spectral methods, dimensionality reduction | 7, 8 |
| **Search, generate, solve** | RL and evolutionary search, generative models, PINNs and operators | 9, 11, 12, 13 |

Underneath all three: reproducibility (Tutorial 1), the variational framing and
generalisation (Tutorial 2), symmetry matched to the data (Tutorials 5, 6, 10), and
independent verification — which is what nearly every section of these thirteen notebooks
has actually been about.

The course perspective, in four questions:

1. What is your task, and what does your data look like?
2. Choose the **method** to match the data.
3. Choose the **architecture** to match the symmetries and structure.
4. Choose the **loss** to match the goal.

### The mini-project

**Deliverables:** a 10-minute presentation (01/10), a five-page report (04/10), and a
reproducible Git repository — environment, seeds, README, code.

**A workflow that works:** one clear question; a dataset you understand; a method matched
to it; a comparison against a simple baseline; an explanation, and independent
verification of whatever the model suggests.

A modest, well-understood result is enough. So is a negative result with a clear
diagnosis — and this tutorial has shown that measuring where a method *fails* is often
more informative than the headline number.

### Further reading

- Raissi, Perdikaris & Karniadakis, "Physics-informed neural networks", *J. Comput. Phys.* **378** (2019).
- Wang, Sankaran & Perdikaris, "Respecting causality for training physics-informed neural networks", *CMAME* **421** (2024) — §2's causal weights.
- Krishnapriyan et al., "Characterizing possible failure modes in physics-informed neural networks", *NeurIPS* 2021 — §2's horizon experiment, done properly.
- Moseley, Markham & Nissen-Meyer, "Finite basis physics-informed neural networks", *Adv. Comput. Math.* **49** (2023) — Exercise 2(d).
- Lu, Jin, Pang, Zhang & Karniadakis, "Learning nonlinear operators via DeepONet", *Nat. Mach. Intell.* **3** (2021) — §4.
- Li et al., "Fourier neural operator for parametric partial differential equations", *ICLR* 2021 — §4's spectral layer.
- Gage & Hamilton, "The heat equation shrinking convex plane curves", *J. Diff. Geom.* **23** (1986); Grayson, "The heat equation shrinks embedded plane curves to round points", *J. Diff. Geom.* **26** (1987) — §3's theorems.

### Boa sorte with the projects!